# Fine-tune MiniLM for movie keyword search

Trains `all-MiniLM-L6-v2` to match keywords to movie text, then compares it with BM25 and the stock model on movies it never trained on.

1. Select a GPU: Runtime > Change runtime type > T4 GPU.
2. Run `python -m moviebot.import_tmdb`, then copy `moviebot.db`, `tags.csv` and `links.csv` from the repo's `data/` folder to `MyDrive/moviebot/`.
3. Runtime > Run all, and allow Drive access.

The model and `results.json` are saved to `MyDrive/moviebot/minilm-movie-search/`.

In [ ]:
!pip install -q sentence-transformers==6.2.0 transformers==5.22.2 datasets==5.2.0 accelerate==1.15.0

from google.colab import drive
drive.mount('/content/drive')

## Settings
Changing these after seeing test results makes the test numbers optimistic.

In [ ]:
import csv, json, random, re, shutil, sqlite3
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then run all again.'

DATA = Path('/content/drive/MyDrive/moviebot')
OUT = DATA / 'minilm-movie-search'
BASE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

MOVIELENS_TAGS = True  # False trains on TMDB keywords only; MovieLens genome scores still grade the test
SEED = 42
TEST_SHARE = 0.2    # share of movies and of tags held out
RELEVANT = 0.8      # genome score at which a tag counts as fitting a movie
MIN_USERS = 2       # users who must give a movie the same tag, to drop personal labels like 'bd-r'
MAX_TAG_CHARS = 40  # longer user tags are mostly full sentences
MAX_PAIRS = None    # training pairs to use; None uses all
EPOCHS = 3
BATCH_SIZE = 64
LEARNING_RATE = 2e-5

needed = ['moviebot.db'] + (['tags.csv', 'links.csv'] if MOVIELENS_TAGS else [])
missing = [f for f in needed if not (DATA / f).exists()]
assert not missing, f'Missing from {DATA}: {missing}'

## Load data and split
Test movies are never trained on. Tags are split too: `unseen tags` never appear in training.

In [ ]:
# SQLite can fail on Drive's filesystem, so read a local copy.
shutil.copy(DATA / 'moviebot.db', '/tmp/moviebot.db')
db = sqlite3.connect('file:/tmp/moviebot.db?mode=ro', uri=True)
assert 'keywords' in {r[1] for r in db.execute('PRAGMA table_info(movies)')}, 'No keywords: run python -m moviebot.import_tmdb, then upload moviebot.db again'

tag_names = [t for (t,) in db.execute('SELECT tag FROM genome_tags ORDER BY tag_id')]
genome = {m: np.frombuffer(b, dtype='f4') for m, b in db.execute('SELECT movie_id, relevance FROM genome')}

titles, texts = {}, {}
for m, title, year, genres, overview in db.execute("SELECT movie_id, title, year, genres, overview FROM movies WHERE overview <> ''"):
    titles[m] = title
    # Matches moviebot.retrieval.movie_document with no tags. Tags stay out because they are the answer key.
    texts[m] = '. '.join(p for p in [f'{title} ({year})', ', '.join((genres or '').split(',')), overview] if p)

rng = random.Random(SEED)
graded = sorted(m for m in genome if m in texts)
rng.shuffle(graded)
test_movies = sorted(graded[:int(len(graded) * TEST_SHARE)])
train_movies = set(texts) - set(test_movies)


def tag_key(tag):
    # shortcut: catches spelling variants only ('zombie', 'Zombies'), so synonyms can still cross the split.
    return re.sub(r'[^a-z0-9]', '', tag.lower()).rstrip('s')


keys = sorted({tag_key(t) for t in tag_names})
test_keys = set(rng.sample(keys, int(len(keys) * TEST_SHARE)))
print(f'{len(train_movies):,} train movies, {len(test_movies):,} test movies, {len(test_keys):,} tags held out')

## Build training pairs
Each pair is a tag and the text of a movie it fits: from MovieLens genome scores and user tags, and from TMDB keywords.

In [ ]:
sources = {}
if MOVIELENS_TAGS:
    sources['genome'] = {
        (tag_names[i], movie)
        for movie, scores in genome.items() if movie in train_movies
        for i in np.flatnonzero(scores >= RELEVANT) if tag_key(tag_names[i]) not in test_keys
    }
    with open(DATA / 'links.csv', newline='', encoding='utf-8') as f:
        imdb_id = {r['movieId']: 'tt' + r['imdbId'] for r in csv.DictReader(f)}
    taggers = defaultdict(set)
    with open(DATA / 'tags.csv', newline='', encoding='utf-8') as f:
        for r in csv.DictReader(f):
            tag, movie = r['tag'].lower().strip(), imdb_id.get(r['movieId'])
            if movie in train_movies and 0 < len(tag) <= MAX_TAG_CHARS and tag_key(tag) not in test_keys:
                taggers[tag, movie].add(r['userId'])
    sources['user tags'] = {pair for pair, users in taggers.items() if len(users) >= MIN_USERS}
sources['TMDB keywords'] = {
    (keyword, movie)
    for movie, keywords in db.execute("SELECT movie_id, keywords FROM movies WHERE keywords <> ''") if movie in train_movies
    for keyword in map(str.strip, keywords.split('|')) if keyword and tag_key(keyword) not in test_keys
}

pairs = sorted(set().union(*sources.values()))
random.Random(SEED).shuffle(pairs)
pairs = pairs[:MAX_PAIRS]
assert not {m for _, m in pairs} & set(test_movies), 'test movie leaked into training'
assert not {tag_key(t) for t, _ in pairs} & test_keys, 'held-out tag leaked into training'
print(', '.join(f'{len(p):,} {name}' for name, p in sources.items()) + f'; {len(pairs):,} pairs used')
print(rng.sample([(t, titles[m]) for t, m in pairs], 5))

## Baselines
A movie is correct for a tag when its genome score reaches `RELEVANT`. P@10 is the correct share of the top 10. R@100 is the share of correct movies found in the top 100.

In [ ]:
from sentence_transformers import SentenceTransformer

truth = np.stack([genome[m] for m in test_movies]) >= RELEVANT
test_docs = [texts[m] for m in test_movies]
# Tags fitting 5 to 100 test movies: fewer is noise, more is a word like 'original' that nobody searches for.
usable = [t for t, n in enumerate(truth.sum(0)) if 5 <= n <= 100]
groups = {
    'seen tags': [t for t in usable if tag_key(tag_names[t]) not in test_keys],
    'unseen tags': [t for t in usable if tag_key(tag_names[t]) in test_keys],
}
query_tags = groups['seen tags'] + groups['unseen tags']
queries = [tag_names[t] for t in query_tags]
results = []


def evaluate(name, ranked):
    """`ranked` holds test-movie positions, best first, for each entry of `query_tags`."""
    by_tag = dict(zip(query_tags, ranked))
    for group, tags in groups.items():
        p10 = float(np.mean([truth[by_tag[t][:10], t].sum() / 10 for t in tags]))
        r100 = float(np.mean([truth[by_tag[t][:100], t].sum() / truth[:, t].sum() for t in tags]))
        results.append({'model': name, 'tags': group, 'queries': len(tags), 'P@10': round(p10, 3), 'R@100': round(r100, 3)})
        print(f'{name:14} {group:12} {len(tags):4} queries   P@10 {p10:.3f}   R@100 {r100:.3f}')


def embed(model, sentences):
    return model.encode(sentences, batch_size=256, normalize_embeddings=True, show_progress_bar=False)


def search(model, docs, sentences, k=100):
    return np.argsort(-(embed(model, sentences) @ docs.T), axis=1)[:, :k]


fts = sqlite3.connect(':memory:')
fts.execute("CREATE VIRTUAL TABLE d USING fts5(doc, tokenize='porter')")
fts.executemany('INSERT INTO d(rowid, doc) VALUES (?, ?)', enumerate(test_docs, 1))
bm25_ranked = []
for q in queries:
    match = ' OR '.join(f'"{w}"' for w in re.findall(r'\w+', q.lower()))
    rows = fts.execute('SELECT rowid FROM d WHERE d MATCH ? ORDER BY bm25(d) LIMIT 100', (match,))
    bm25_ranked.append(np.array([r - 1 for (r,) in rows], dtype=int))
evaluate('BM25', bm25_ranked)

stock = SentenceTransformer(BASE_MODEL)
stock_docs = embed(stock, test_docs)
evaluate('stock MiniLM', search(stock, stock_docs, queries))

## Train

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss

model = SentenceTransformer(BASE_MODEL)
train_data = Dataset.from_dict({'anchor': [t for t, _ in pairs], 'positive': [texts[m] for _, m in pairs]})
args = SentenceTransformerTrainingArguments(
    output_dir='/tmp/checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=0.1,
    batch_sampler='no_duplicates',
    fp16=torch.cuda.is_available(),
    save_strategy='no',
    logging_steps=200,
    report_to='none',
    seed=SEED,
)
# shortcut: other movies in a batch count as non-matches even when some do fit the tag. Mask them with a custom loss if results stall.
loss = MultipleNegativesRankingLoss(model)
SentenceTransformerTrainer(model=model, args=args, train_dataset=train_data, loss=loss).train()
model.save_pretrained(str(OUT))

## Results

In [ ]:
tuned = SentenceTransformer(str(OUT))
tuned_docs = embed(tuned, test_docs)
evaluate('tuned MiniLM', search(tuned, tuned_docs, queries))
(OUT / 'results.json').write_text(json.dumps({'train_pairs': len(pairs), 'results': results}, indent=2))

## Sample searches
Searches run over the test movies only. The last three queries are movie titles, to check title lookup still works.

In [ ]:
samples = ['feel-good', 'twist ending', 'cyberpunk', 'time travel', 'heist', 'zombies', 'movies that make you cry', 'robots taking over the world']
samples += [titles[m] for m in random.Random(SEED).sample(test_movies, 3)]
stock_top, tuned_top = search(stock, stock_docs, samples, 5), search(tuned, tuned_docs, samples, 5)
for query, s, t in zip(samples, stock_top, tuned_top):
    print(f'[{query}]')
    print('  stock:', '; '.join(titles[test_movies[i]] for i in s))
    print('  tuned:', '; '.join(titles[test_movies[i]] for i in t))